# lab09 · DriveVLM-Dual 的"快慢双系统"最小 pipeline
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChatGPU/Autonomous-Driving-Learning-Atlas/blob/main/labs/lab09_drivevlm_dual_pipeline.ipynb)

**配套节点**：[DriveVLM](../docs/data/cards/paper_2402.12289_drivevlm.md)

**What this proves**：把 VLM 作为"慢系统"做高层 meta-action 输出，再用 IDM 作"快系统"
翻译成轨迹；遇到日常场景跳过慢系统，遇到长尾才启用——大幅压缩端到端时延。


In [1]:
import sys, json, time
sys.path.insert(0, "..")
from labs.llm_provider import LLM
llm = LLM()

# IDM (Intelligent Driver Model) as fast planner
def idm_acc(v_ego, v_lead, gap, v0=27.0, T=1.5, a=1.5, b=2.0, s0=2.0):
    s_star = s0 + max(0, v_ego*T + (v_ego*(v_ego - v_lead))/(2*(a*b)**0.5))
    return a*(1 - (v_ego/v0)**4 - (s_star/max(gap,1e-3))**2)

def fast_planner(scene):
    v_ego = scene["ego_speed"]; v_lead = scene.get("lead_speed", 25.0); gap = scene["lead_distance"]
    acc = idm_acc(v_ego, v_lead, gap)
    return {"acc_mps2": acc, "type": "fast"}

def slow_planner_vlm(scene):
    sys_msg = "你是 DriveVLM。基于 scene 做三步 CoT：场景描述/分析/决策。仅返回 JSON {meta_action, ...}."
    user = f"场景 JSON: {json.dumps(scene, ensure_ascii=False)}"
    return llm.json_chat([{"role":"system","content":sys_msg},{"role":"user","content":user}])

def is_easy(scene):
    return scene["lead_distance"] > 35 and scene.get("weather","clear") == "clear"

def dual_planner(scene):
    if is_easy(scene): return fast_planner(scene), None
    meta = slow_planner_vlm(scene)
    plan = fast_planner(scene)
    plan["overrided_by_vlm"] = meta.get("meta_action")
    return plan, meta

scenes = [
    {"name":"easy",  "lead_distance": 60.0, "lead_speed": 26.0, "ego_speed": 25.0, "weather": "clear"},
    {"name":"close", "lead_distance": 8.0,  "lead_speed": 14.0, "ego_speed": 26.0, "weather": "rain"},
    {"name":"empty", "lead_distance": 80.0, "lead_speed": 28.0, "ego_speed": 22.0, "weather": "clear"},
]
for s in scenes:
    t = time.time()
    plan, meta = dual_planner(s)
    print(f"[{s['name']}] decided in {(time.time()-t)*1000:.1f}ms  plan={plan}  meta={meta}")

assert any(s["name"]=="easy" and dual_planner(s)[1] is None for s in scenes), "easy scene must skip VLM"
print("PASS — Dual planner skips VLM on easy scenes and engages on long-tail")


[easy] decided in 0.0ms  plan={'acc_mps2': -0.03679475523992087, 'type': 'fast'}  meta=None
[close] decided in 0.2ms  plan={'acc_mps2': -402.41008652913894, 'type': 'fast', 'overrided_by_vlm': 'decelerate'}  meta={'meta_action': 'decelerate', 'rationale': '前车距离 8.0m 过近，先减速保持安全距离', 'target_speed': 23.0, 'target_lane': 1}
[empty] decided in 0.0ms  plan={'acc_mps2': 0.837871511724726, 'type': 'fast'}  meta=None
PASS — Dual planner skips VLM on easy scenes and engages on long-tail


### 三个 stretch goals
1. 用 `transformers` 加载 `Qwen/Qwen2-VL-2B-Instruct`，把 scene 换成真实图像 + caption；
2. 把 `is_easy` 阈值学出来（一个 small classifier），重现 [CF-VLA 的 adaptive thinking](../docs/data/cards/paper_2512.24426_cfvla.md) 思想；
3. 收集 20 个 scene，量化 VLM 启用比例 vs 安全指标。
